# 05-10 Fallbacks: 오류 발생 시 자동으로 대체 경로 사용하기

LLM API는 네트워크를 통해 호출하기 때문에 다양한 이유로 실패할 수 있어요. Rate Limit, 서버 장애, 일시적 네트워크 오류 등이 대표적이에요. 프로덕션 환경에서는 "기본 모델이 실패하면 대체 모델로 자동 전환"하는 안전망이 필수예요.

LangChain의 `with_fallbacks()` 메서드가 이 역할을 해요.

## 학습 목표

- `with_fallbacks()`로 기본 체인이 실패했을 때 자동으로 대체 체인으로 전환하는 폴백(Fallback) 패턴을 구현할 수 있어요
- 여러 폴백 체인을 순차적으로 지정하여 단계적 복구 전략을 설계할 수 있어요
- `exceptions_to_handle`로 특정 예외만 폴백으로 처리하고 나머지는 그대로 발생시키는 방법을 설명할 수 있어요

## 사전 지식

- 파이썬 예외 처리(`try/except`) 기초
- `ChatOpenAI` 모델 초기화와 `max_retries` 파라미터
- LCEL 파이프 연산자(`|`) 사용법

---

### 오류 복구 전략: 재시도 vs 폴백

LLM 호출이 실패했을 때 두 가지 복구 전략이 있어요:

| 전략 | 동작 | 적합한 상황 |
|------|------|------------|
| **재시도(Retry)** | 같은 모델로 다시 시도 | 일시적 네트워크 오류, 서버 과부하 |
| **폴백(Fallback)** | 다른 모델로 전환 | 특정 모델 장애, Rate Limit |

LangChain에서는 `max_retries`로 재시도를, `with_fallbacks()`로 폴백을 설정해요. 두 전략을 함께 사용할 수도 있지만, 아래에서 설명하듯 의도치 않은 중복에 주의해야 해요.

> 🔑 **핵심 개념**: `with_fallbacks()`는 주 모델이 실패했을 때 대체 모델로 자동 전환하는 안전망이에요.

아래 다이어그램은 폴백 체인의 실행 흐름을 보여줘요:

```mermaid
flowchart TD
    IN["입력"]:::input
    P["Primary 모델<br/>기본 실행"]:::process
    F1["Fallback 1<br/>첫 번째 대체 모델"]:::error
    F2["Fallback 2<br/>두 번째 대체 모델"]:::error
    F3["Fallback 3<br/>세 번째 대체 모델"]:::error
    OK["성공<br/>응답 반환"]:::output
    ERR["최종 실패<br/>예외 발생"]:::error

    IN --> P
    P -- "성공" --> OK
    P -- "실패" --> F1
    F1 -- "성공" --> OK
    F1 -- "실패" --> F2
    F2 -- "성공" --> OK
    F2 -- "실패" --> F3
    F3 -- "성공" --> OK
    F3 -- "실패" --> ERR

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
```

**중요:** 폴백은 LLM 수준뿐만 아니라 전체 실행 가능한 체인(`prompt | llm | parser`) 수준에도 적용할 수 있어요.

In [ ]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 기본 모델 (재시도 비활성화)
primary_llm = ChatOpenAI(max_retries=0, model="gpt-4o-mini")

# Fallback 모델 (다른 모델 또는 서비스)
# 예제에서는 동일한 모델을 사용하지만, 실제로는 다른 모델을 사용
fallback_llm = ChatOpenAI(max_retries=0, model="gpt-3.5-turbo")

print("=" * 60)
print("✅ 모델 초기화 완료")
print("=" * 60)


## 1. 기본 Fallback 설정

`with_fallbacks()`를 사용하여 기본 모델이 실패할 때 대체 모델을 자동으로 사용하도록 설정해요.

### max_retries=0을 권장하는 이유

LangChain의 `ChatOpenAI`는 기본적으로 `max_retries=2`로 설정되어 있어요. 이는 실패 시 같은 모델로 2번 재시도한다는 뜻이에요. 여기에 `with_fallbacks()`까지 추가하면 이런 흐름이 돼요:

```
기본 모델 시도 → 실패 → 재시도 1 → 실패 → 재시도 2 → 실패 → 폴백 모델 시도
```

재시도마다 대기 시간이 추가되므로 폴백 전환이 느려져요. `max_retries=0`으로 설정하면 즉시 폴백으로 전환할 수 있어요.

> 🎯 **강의 포인트**: `max_retries=0`으로 설정해야 fallback이 즉시 작동해요. 기본값은 재시도를 먼저 해요.

> 💡 **실무 팁**: 프로덕션에서는 "GPT-4o -> GPT-4o-mini -> Claude" 같은 다단계 폴백을 구성해요. 고성능 모델이 실패하면 가벼운 모델이라도 응답을 보장하는 거예요.

In [ ]:
# ============================================================
# TODO: primary_llm.with_fallbacks([fallback_llm])으로 폴백을 설정하세요
# 힌트: max_retries=0 권장 — 재시도와 폴백 로직이 중복되지 않도록
# 예상 결과: 기본 모델 실패 시 fallback_llm이 자동으로 실행
# ============================================================

# ---------------------------------------------------
# with_fallbacks([...]): 기본 체인 실패 시 대체 체인 순차 시도
# ---------------------------------------------------

# 1단계: 폴백 설정
# max_retries=0: LangChain의 내부 재시도를 비활성화
# - 폴백 로직과 재시도 로직이 중복되면 불필요한 대기 시간 발생
# - with_fallbacks()로 명시적인 재시도 전략 구성
llm_with_fallback = primary_llm.with_fallbacks([fallback_llm])

print("=" * 60)
print("✅ Fallback 설정 완료")
print("=" * 60)
print("기본 모델 실패 시 fallback 모델이 자동으로 사용됩니다.")


In [ ]:
# Fallback이 설정된 모델 실행
# 
# 실제 오류가 발생하지 않으므로 정상 실행됨
result = llm_with_fallback.invoke("대한민국의 수도는 어디야?")

print("=" * 60)
print("📥 Fallback 모델 실행")
print("=" * 60)
print(result.content)
print()
print("💡 기본 모델이 정상 작동하면 fallback은 사용되지 않음")


## 2. 특정 예외만 처리하기

### 복구 가능한 예외 vs 치명적 예외

모든 예외를 폴백으로 처리하면 안 돼요. 예외의 성격에 따라 처리 방식이 달라야 해요.

| 예외 유형 | 예시 | 폴백 적합? | 이유 |
|-----------|------|----------|------|
| Rate Limit | `RateLimitError` | O | 다른 모델은 호출 가능 |
| 서버 오류 | `APIStatusError(500)` | O | 일시적 장애, 다른 모델 시도 |
| 인증 실패 | `AuthenticationError` | X | API 키 문제, 폴백해도 같은 오류 |
| 잘못된 모델명 | `NotFoundError` | X | 설정 오류, 코드를 수정해야 함 |

`exceptions_to_handle` 파라미터를 사용하면 특정 예외만 폴백으로 처리할 수 있어요. 나머지 예외는 그대로 발생시켜서 문제의 근본 원인을 파악할 수 있게 해요.

> ⚠️ **주의**: 모든 예외를 잡는 것보다 특정 예외만 처리하는 게 안전해요. `exceptions_to_handle`로 범위를 좁히세요.

> 🔑 **핵심 개념**: 모든 예외를 폴백으로 처리하면 실제 오류(API 키 누락, 잘못된 모델명 등)가 묻혀버릴 수 있어요. `exceptions_to_handle`로 폴백을 트리거할 예외 범위를 좁히는 것이 프로덕션에서 올바른 패턴이에요.

In [ ]:
# 특정 예외만 처리하는 Fallback 설정
# 
# 시나리오: KeyboardInterrupt만 fallback으로 처리
#           다른 예외는 그대로 발생시킴

llm_selective_fallback = primary_llm.with_fallbacks(
    [fallback_llm],
    exceptions_to_handle=(KeyboardInterrupt,),  # 특정 예외만 처리
)

print("=" * 60)
print("✅ 선택적 Fallback 설정 완료")
print("=" * 60)
print("KeyboardInterrupt만 fallback으로 처리됩니다.")


## 3. 여러 Fallback 모델 순차 지정

여러 개의 폴백 모델을 지정하면 순차적으로 시도해요.

```
기본 모델 실패 → 폴백 1 시도 → 실패 → 폴백 2 시도 → 실패 → 폴백 3 시도 → ...
```

### 폴백 적용 범위

폴백은 다양한 수준에서 적용할 수 있어요:

| 적용 수준 | 코드 예시 | 효과 |
|-----------|-----------|------|
| **모델 수준** | `llm.with_fallbacks([fallback_llm])` | LLM만 교체 |
| **체인 수준** | `chain.with_fallbacks([fallback_chain])` | 프롬프트 + LLM + 파서 전체 교체 |

모델 수준 폴백은 같은 프롬프트를 다른 모델에 전달해요. 체인 수준 폴백은 프롬프트부터 파서까지 완전히 다른 체인으로 교체할 수 있어요. 예를 들어, OpenAI 체인이 실패하면 Anthropic 체인으로 전환하는 식이에요.

> ⚠️ **주의**: 폴백 목록에서 비용이 낮은 모델을 앞에, 신뢰성이 높은 모델을 뒤에 배치하세요. 비용과 신뢰성 사이의 균형을 고려해서 폴백 순서를 설계해야 해요.

> 💡 **실무 팁**: 폴백 체인에서 각 모델에 `max_retries=0`을 설정하면 실패 감지가 빨라요. 모델마다 재시도를 기다리면 전체 폴백 체인 실행 시간이 크게 늘어날 수 있어요.

In [ ]:
# ============================================================
# TODO: 여러 fallback 모델을 순차적으로 지정하는 체인을 구성하세요
# 힌트: bad_model.with_fallbacks([fallback1, fallback2, fallback3])
# 예상 결과: bad_model 실패 → fallback_chain1 시도 → 성공 시 결과 반환
# ============================================================

# ---------------------------------------------------
# 여러 폴백 모델: 순차 시도로 단계적 복구
# ---------------------------------------------------

# 1단계: 잘못된 모델명으로 반드시 실패하는 모델 (시뮬레이션용)
# max_retries=0: 즉시 실패하여 폴백으로 빠르게 전환
bad_model = ChatOpenAI(model="gpt-fake-model", max_retries=0)

# 2단계: 여러 폴백 모델 정의 (순서대로 시도)
# 앞: 비용 낮은 모델, 뒤: 더 신뢰할 수 있는 모델 배치 권장
fallback_chain1 = ChatOpenAI(model="gpt-3.5-turbo", max_retries=0)
fallback_chain2 = ChatOpenAI(model="gpt-4o-mini", max_retries=0)
fallback_chain3 = ChatOpenAI(model="gpt-4o", max_retries=0)

# 3단계: with_fallbacks()로 여러 폴백을 순차 지정
# - bad_model 실패 → fallback_chain1 시도
# - fallback_chain1 실패 → fallback_chain2 시도
# - fallback_chain2 실패 → fallback_chain3 시도
# - 모든 폴백 실패 → 예외 발생
chain_with_multiple_fallbacks = bad_model.with_fallbacks([
    fallback_chain1,
    fallback_chain2,
    fallback_chain3
])

print("=" * 60)
print("✅ 여러 Fallback 설정 완료")
print("=" * 60)
print("실패 시 순차적으로 fallback 모델을 시도합니다:")
print("  1. bad_model (실패 예상)")
print("  2. fallback_chain1")
print("  3. fallback_chain2")
print("  4. fallback_chain3")


In [ ]:
# 프롬프트와 함께 사용하는 예제
prompt = ChatPromptTemplate.from_template(
    "질문에 짧고 간결하게 답변해 주세요.\n\n질문: {question}\n\n답변:"
)

# 체인에 fallback 적용
chain = prompt | chain_with_multiple_fallbacks

# 실행 (실제로는 첫 번째 fallback이 성공할 것)
try:
    result = chain.invoke({"question": "대한민국의 수도는 어디야?"})
    print("=" * 60)
    print("📥 여러 Fallback 체인 실행")
    print("=" * 60)
    print(result.content)
except Exception as e:
    print(f"오류 발생: {e}")


## 정리

### with_fallbacks() 주요 파라미터

| 파라미터 | 역할 | 예시 |
|----------|------|------|
| 첫 번째 인자 (리스트) | 순서대로 시도할 폴백 체인 목록 | `[fallback1, fallback2]` |
| `exceptions_to_handle` | 폴백을 트리거할 예외 타입 | `(RateLimitError,)` |

### 폴백 설계 체크리스트

- [ ] 기본 모델에 `max_retries=0` 설정했는지 확인
- [ ] `exceptions_to_handle`로 복구 가능한 예외만 지정했는지 확인
- [ ] 폴백 순서가 비용/성능 기준으로 적절한지 확인
- [ ] 모든 폴백이 실패할 경우의 에러 처리가 있는지 확인

### 핵심 요점

- `with_fallbacks([...])`로 기본 체인이 실패했을 때 자동으로 대체 체인을 순차적으로 시도해요
- 기본 모델에 `max_retries=0`을 설정하면 폴백 로직과 재시도 로직의 충돌을 방지하고 즉시 전환할 수 있어요
- `exceptions_to_handle`로 특정 예외만 폴백으로 처리하고 나머지는 그대로 전파할 수 있어요
- 폴백은 모델 수준뿐 아니라 전체 체인(`prompt | llm | parser`) 수준에도 적용할 수 있어요
- 폴백 목록의 앞에는 비용이 낮은 모델, 뒤에는 신뢰성이 높은 모델을 배치하세요

### 05_Advanced-LCEL 전체 요약

이 챕터에서 배운 LCEL 고급 기능들을 정리해요:

| 노트북 | 핵심 기능 | 주요 사용 사례 |
|--------|-----------|----------------|
| 01 RunnablePassthrough | 데이터 흐름 유지 + 키 추가 | RAG 질문 전달 |
| 02 Inspect Runnables | 체인 구조 시각화 | 디버깅, 구조 확인 |
| 03 RunnableLambda/@chain | 함수를 체인으로 변환 | 전처리/후처리 로직 |
| 04 Routing | 조건부 분기 | 다분야 챗봇 |
| 05 RunnableParallel | 병렬 실행 | 성능 최적화 |
| 06 Configure | 런타임 설정 변경 | A/B 테스트, 다중 모델 |
| 07 MessageHistory | 대화 기록 관리 | 멀티턴 챗봇 |
| 08 Custom Generator | 스트리밍 데이터 변환 | 실시간 출력 파서 |
| 09 Binding | 파라미터/도구 사전 연결 | Tool Calling |
| 10 Fallbacks | 오류 복구 전략 | 안정적 서비스 |